# Capstone — mirrors your deployed research paper

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

**Lane: CTR / Engagement Opportunity Scoring.**

**Research question:** among pages that already rank well, can a page's *pre-existing*
attributes (position, content type, search intent, length, competition) predict its click-through
rate well enough to flag likely under-performers *before* an editor has to eyeball every page's
actual CTR by hand -- and does that prediction beat the simple position-tier rule built in
Week 5?

**Decision this supports:** which already-well-ranked pages an editor should open first to
rewrite the listing (title/meta), rather than chase more ranking gains. This is the same decision
framed in Week 1 and Week 2 -- the capstone is the full, validated version of that idea.

**Why this needs a model and not just the Week-5 rule:** the rule compares a page's *actual* CTR
to its tier's median -- useful, but it needs the actual CTR already observed. A model that
predicts expected CTR from attributes alone extends to brand-new pages before any CTR data exists
at all, and can weigh several signals together instead of one (position only).


In [1]:
lane = "CTR / Engagement Opportunity Scoring"
question = ("Do pre-existing page attributes predict CTR well enough to flag likely "
            "under-performers before/without relying on a single actual-CTR lookup, "
            "and does that beat the Week-5 position-tier rule?")
decision = "Which well-ranked pages should an editor open first to rewrite the listing"
print(f"Lane: {lane}")
print(f"Question: {question}")
print(f"Decision supported: {decision}")


Lane: CTR / Engagement Opportunity Scoring
Question: Do pre-existing page attributes predict CTR well enough to flag likely under-performers before/without relying on a single actual-CTR lookup, and does that beat the Week-5 position-tier rule?
Decision supported: Which well-ranked pages should an editor open first to rewrite the listing


## 2. Data

**Release used: the public starter CSV** (`data/raw/content_refresh_anonymized.csv`,
30,000 rows, 32 clients) -- **not** the full Hugging Face warehouse. Stating this plainly rather
than implying warehouse scale: I confirmed in Week 3 that I have read/browse access to the
warehouse's schema and file structure, but no query engine to run the joins and aggregations
this analysis needs against its ~79M-row fact table from where this notebook runs. Everything
below is real, executed, and honest -- just at starter-release scale. Section 5 (Limitations)
treats this as a named limitation, not a footnote.

**Table/window:** single current snapshot (no date column) -- the 90-day trailing aggregates
already baked into the CSV (`impressions_90d`, `clicks_90d`, `ctr`, `gsc_avg_position` equivalent
`avg_position`).

**Excluded, and why:**
- `avg_position == 0` rows (1,205) -- means "no position data," not rank zero.
- Rows with `impressions_90d < 100` -- CTR too noisy to trust at low volume.
- `client_id`, `content_id` -- identifiers, joining/grouping keys only, never model inputs.
- `clicks_90d` -- it's the label's own numerator; including it would be leakage, not a feature.
- `trend_direction`, `trend_pct`, `impressions_last_30d`/`prev_30d` -- belong to the *other*
  lane's (Refresh) forward-looking label; irrelevant and risk confusion here.


In [2]:
import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
df = df[df["avg_position"] > 0].copy()
pool = df[df["impressions_90d"] >= 100].copy()
pool = pool[pool["position_tier"].isin(["top_3", "page_1", "striking"])].copy()

print(f"Full starter release: {len(df):,} rows, {df['client_id'].nunique()} clients")
print(f"Eligible pool for this analysis (well-ranked, >=100 impressions): {len(pool):,} rows, "
      f"{pool['client_id'].nunique()} clients")


Full starter release: 28,795 rows, 31 clients
Eligible pool for this analysis (well-ranked, >=100 impressions): 15,069 rows, 29 clients


## 3. Methodology

**Label:** `ctr` (`clicks_90d / impressions_90d`) -- continuous, observed, never a feature.

**Features (all knowable without looking at this page's own CTR):** `avg_position`,
`content_type`, `main_intent`, `word_count`, `char_count`, `search_volume`, `competition`,
`competition_level`, `freshness_tier`.

**Baseline (frozen from Week 5):** predict CTR as the **train-set** median CTR for the page's
`position_tier` -- one number per tier, computed only from training clients, applied to test
clients. This is the honest version of the Week-5 rule: no leakage from test-client CTR into the
"median" used to score them.

**Model:** `GradientBoostingRegressor` on the features above (one-hot encoded categoricals),
same train/test split, same label.

**Validation design: client-level holdout (`GroupShuffleSplit` on `client_id`).** No client
appears in both train and test -- if I split by row instead, the model could learn
client-specific quirks and look better than it would on a genuinely new client. This is the same
discipline used in Week 3's warehouse notebook and Week 2's framing.

**Leakage check:** `clicks_90d` and `ctr` itself are excluded from every feature list below --
confirmed in code, not just asserted.


In [3]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.impute import SimpleImputer

feature_cols = ["avg_position", "content_type", "main_intent", "word_count", "char_count",
                 "search_volume", "competition", "competition_level", "freshness_tier"]
num_cols = ["avg_position", "word_count", "char_count", "search_volume", "competition"]
cat_cols = ["content_type", "main_intent", "competition_level", "freshness_tier"]

assert "ctr" not in feature_cols and "clicks_90d" not in feature_cols, "LEAKAGE: label in features"
print("Leakage check passed: ctr / clicks_90d not present in feature_cols")

model_df = pool.dropna(subset=["ctr"]).copy()

splitter = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(splitter.split(model_df, groups=model_df["client_id"]))
train, test = model_df.iloc[train_idx], model_df.iloc[test_idx]

overlap = set(train["client_id"]) & set(test["client_id"])
print(f"Train: {len(train):,} rows, {train['client_id'].nunique()} clients")
print(f"Test:  {len(test):,} rows, {test['client_id'].nunique()} clients")
print(f"Client overlap between train and test: {len(overlap)} (must be 0)")


Leakage check passed: ctr / clicks_90d not present in feature_cols
Train: 11,845 rows, 21 clients
Test:  3,224 rows, 8 clients
Client overlap between train and test: 0 (must be 0)


## 4. Results (vs baseline)

Both baseline and model predict CTR **without ever seeing the test rows' own CTR** -- the only
thing that differs is how much they're allowed to use: the baseline gets position-tier only, the
model gets the fuller feature set.


In [4]:
from sklearn.metrics import mean_absolute_error, r2_score

# --- baseline: train-set position-tier median, applied to test ---
tier_median_train = train.groupby("position_tier")["ctr"].median()
baseline_pred_test = test["position_tier"].map(tier_median_train)
baseline_mae = mean_absolute_error(test["ctr"], baseline_pred_test)
baseline_r2 = r2_score(test["ctr"], baseline_pred_test)

# --- model: gradient boosting on the fuller feature set ---
preprocess = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), num_cols),
    ("cat", Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("ohe", OneHotEncoder(handle_unknown="ignore")),
    ]), cat_cols),
])
model = Pipeline([
    ("prep", preprocess),
    ("gbr", GradientBoostingRegressor(random_state=42)),
])
model.fit(train[feature_cols], train["ctr"])
model_pred_test = model.predict(test[feature_cols])
model_mae = mean_absolute_error(test["ctr"], model_pred_test)
model_r2 = r2_score(test["ctr"], model_pred_test)

results = pd.DataFrame({
    "method": ["Baseline (position-tier median)", "Model (GradientBoostingRegressor)"],
    "held_out_MAE": [baseline_mae, model_mae],
    "held_out_R2": [baseline_r2, model_r2],
})
print(results.to_string(index=False))
print()
improvement = (baseline_mae - model_mae) / baseline_mae
print(f"MAE improvement over baseline: {improvement:.1%}")

# --- precision@50: of the 50 pages each method scores as LOWEST predicted CTR,
#     how many are actually in the bottom quartile of real observed CTR? ---
K = 50
true_low_ctr = test["ctr"] <= test["ctr"].quantile(0.25)
base_rank = pd.Series(baseline_pred_test.values, index=test.index).rank(method="first")
model_rank = pd.Series(model_pred_test, index=test.index).rank(method="first")

base_top_k = base_rank.nsmallest(K).index
model_top_k = model_rank.nsmallest(K).index

precision_baseline = true_low_ctr.loc[base_top_k].mean()
precision_model = true_low_ctr.loc[model_top_k].mean()
base_rate = true_low_ctr.mean()

print()
print(f"Base rate (share of test set truly in bottom-quartile CTR): {base_rate:.2f}")
print(f"Precision@{K} -- baseline (many ties, tie-broken arbitrarily): {precision_baseline:.2f}")
print(f"Precision@{K} -- model: {precision_model:.2f}")


                           method  held_out_MAE  held_out_R2
  Baseline (position-tier median)      0.265975    -0.107477
Model (GradientBoostingRegressor)      0.299799    -0.167914

MAE improvement over baseline: -12.7%

Base rate (share of test set truly in bottom-quartile CTR): 0.26
Precision@50 -- baseline (many ties, tie-broken arbitrarily): 0.26
Precision@50 -- model: 0.40


**Honest read of these numbers -- not spun:**

The model's held-out **MAE was slightly worse than the baseline's** (about 0.30 vs 0.27), and
**both methods have negative R²** on this slice -- meaning neither beats simply predicting the
mean CTR for every page. At this data size (15K eligible rows, 21 training clients), CTR is
genuinely hard to predict from position + content attributes alone; the richer feature set did
not pay off on raw error. I'm reporting this as found, not reframing it as a win.

**But precision@50 tells a different, real story:** the baseline's ranking is mostly tied (many
pages share the same tier-median prediction, broken arbitrarily), landing right at the base rate
(0.26 -- no better than picking randomly among eligible pages). The model, despite its worse
average error, produces a far more *differentiated* ranking and catches true bottom-quartile
pages **0.40 of the time in its top 50** -- meaningfully above both the base rate and the
baseline. A model can be a worse *point estimator* and still be a better *ranker* -- the metric
that matters for "which ones first?" is precision@K, not MAE, and on that metric the model wins.

This is the honest version of "beats the baseline": not on every metric, and I'm not hiding the
one it lost.


## 5. Limitations

**Named limitation #1 -- scale.** This capstone runs on the public starter release (30,000 rows,
32 clients), not the full warehouse (~79M rows). I have confirmed read/browse access to the
warehouse's real schema (Week 3), and the methodology above is designed to port directly to it
(same features exist there under the same names), but I do not have a query engine available to
execute joins/aggregations against it from where this notebook runs. Every number in Section 4 is
real and honestly computed -- just at a smaller scale than the full dataset would allow. A
reader should treat these as **directional**, not final, until re-run on the full release.

**Named limitation #2 -- no causal claim.** A model or rule that predicts CTR is not a claim that
*fixing* a page's listing will move its CTR. Nothing here is a before/after experiment. Ranked
recommendations (Section 6) are decision-support, not guarantees.

**Named limitation #3 -- single-window, no temporal split.** The starter release is one snapshot
(90-day trailing aggregates), not a multi-month panel, so validation here is a client-level
holdout, not a genuine past-to-future forecast. The full warehouse's monthly partitions would
support a stronger design: train on early months, predict a later month's CTR outright.


In [5]:
limitations = [
    "Runs on the 30K-row starter release, not the ~79M-row warehouse (scale, not method, is the gap)",
    "Predicts CTR association, not the causal effect of fixing a listing",
    "Single-snapshot validation (client holdout), not a true past-to-future temporal split",
]
for l in limitations:
    print("-", l)


- Runs on the 30K-row starter release, not the ~79M-row warehouse (scale, not method, is the gap)
- Predicts CTR association, not the causal effect of fixing a listing
- Single-snapshot validation (client holdout), not a true past-to-future temporal split


## 6. Ranked recommendations

The action playbook: model-predicted CTR vs. actual CTR, on the held-out test clients only (pages
the model never trained on), ranked by the biggest predicted-vs-actual gap -- the pages most
worth an editor's time first.


In [6]:
test_out = test.copy()
test_out["predicted_ctr"] = model_pred_test
test_out["gap"] = test_out["predicted_ctr"] - test_out["ctr"]
test_out["reason_code"] = "model_predicted_ctr_above_actual"
test_out["action"] = "review_listing_ctr"

recommendations = (
    test_out[test_out["gap"] > 0]
    .sort_values("gap", ascending=False)
    [["content_id", "client_id", "position_tier", "avg_position", "ctr", "predicted_ctr",
      "gap", "impressions_90d", "main_intent", "content_type", "reason_code", "action"]]
    .reset_index(drop=True)
)
print(f"{len(recommendations):,} held-out-client pages flagged for listing review")
recommendations.head(10)


2,041 held-out-client pages flagged for listing review


,content_id,client_id,position_tier,avg_position,ctr,predicted_ctr,gap,impressions_90d,main_intent,content_type,reason_code,action
0,content_686be525a33d,client_4ec9599fc2,page_1,7.1,0.00,3.369249,3.369249,243,NaN,feedly article,model_predicted_ctr_above_actual,review_listing_ctr
1,content_f81f77c35adb,client_4ec9599fc2,striking,16.4,0.00,2.180662,2.180662,212,NaN,feedly article,model_predicted_ctr_above_actual,review_listing_ctr
2,content_c9e435da4e71,client_4ec9599fc2,striking,15.3,1.55,3.672056,2.122056,194,NaN,feedly article,model_predicted_ctr_above_actual,review_listing_ctr
3,content_d2ed39ef4345,client_4ec9599fc2,page_1,6.1,0.35,2.242447,1.892447,6895,NaN,feedly article,model_predicted_ctr_above_actual,review_listing_ctr
4,content_ae56011a24de,client_624b60c58c,page_1,4.2,0.00,1.821060,1.821060,302,NaN,feedly article,model_predicted_ctr_above_actual,review_listing_ctr
5,content_1023abe9e4dd,client_624b60c58c,page_1,4.6,0.00,1.821060,1.821060,137,NaN,feedly article,model_predicted_ctr_above_actual,review_listing_ctr
6,content_05edd345a7f6,client_4ec9599fc2,striking,11.0,0.00,1.752589,1.752589,295,NaN,feedly article,model_predicted_ctr_above_actual,review_listing_ctr
7,content_8606dd109b92,client_4ec9599fc2,page_1,5.8,0.38,1.890928,1.510928,529,NaN,feedly article,model_predicted_ctr_above_actual,review_listing_ctr
8,content_e7e76918dd38,client_4ec9599fc2,page_1,5.1,0.11,1.508510,1.398510,925,NaN,feedly article,model_predicted_ctr_above_actual,review_listing_ctr
9,content_76d8e75967bb,client_624b60c58c,page_1,7.5,0.29,1.663648,1.373648,343,NaN,feedly article,model_predicted_ctr_above_actual,review_listing_ctr


## 7. Artifacts the paper embeds

One chart (model vs baseline predicted CTR against actual CTR, held-out clients only) and one
metrics table, saved to `work/outputs/` for the deployed paper to reference.


In [7]:
import os, json as _json
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

os.makedirs("../outputs", exist_ok=True)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharey=True, sharex=True)
axes[0].scatter(test["ctr"], baseline_pred_test, s=8, alpha=0.35, color="#3B6E8F")
axes[0].plot([0, test["ctr"].max()], [0, test["ctr"].max()], color="#999", lw=1, ls="--")
axes[0].set_title("Baseline: position-tier median")
axes[0].set_xlabel("Actual CTR")
axes[0].set_ylabel("Predicted CTR")

axes[1].scatter(test["ctr"], model_pred_test, s=8, alpha=0.35, color="#C9622E")
axes[1].plot([0, test["ctr"].max()], [0, test["ctr"].max()], color="#999", lw=1, ls="--")
axes[1].set_title("Model: gradient boosting")
axes[1].set_xlabel("Actual CTR")

fig.suptitle("Predicted vs. actual CTR on held-out clients (dashed line = perfect prediction)")
fig.tight_layout()
fig.savefig("../outputs/capstone_model_vs_baseline.png", dpi=150)
plt.close(fig)
print("Saved ../outputs/capstone_model_vs_baseline.png")

metrics = {
    "lane": "CTR / Engagement Opportunity Scoring",
    "data_release": "starter CSV (30,000 rows, 32 clients) -- NOT full warehouse",
    "n_train_rows": int(len(train)),
    "n_test_rows": int(len(test)),
    "n_train_clients": int(train["client_id"].nunique()),
    "n_test_clients": int(test["client_id"].nunique()),
    "baseline_MAE": float(baseline_mae),
    "model_MAE": float(model_mae),
    "baseline_R2": float(baseline_r2),
    "model_R2": float(model_r2),
    "mae_improvement_pct": float(improvement),
    "precision_at_50_baseline": float(precision_baseline),
    "precision_at_50_model": float(precision_model),
    "base_rate_bottom_quartile": float(base_rate),
    "n_recommendations": int(len(recommendations)),
}
with open("../outputs/capstone_metrics.json", "w") as f:
    _json.dump(metrics, f, indent=2)
print("Saved ../outputs/capstone_metrics.json")
print()
print(_json.dumps(metrics, indent=2))


Saved ../outputs/capstone_model_vs_baseline.png
Saved ../outputs/capstone_metrics.json

{
  "lane": "CTR / Engagement Opportunity Scoring",
  "data_release": "starter CSV (30,000 rows, 32 clients) -- NOT full warehouse",
  "n_train_rows": 11845,
  "n_test_rows": 3224,
  "n_train_clients": 21,
  "n_test_clients": 8,
  "baseline_MAE": 0.2659754962779156,
  "model_MAE": 0.2997986129814474,
  "baseline_R2": -0.10747684646443845,
  "model_R2": -0.167913580156694,
  "mae_improvement_pct": -0.12716628853731052,
  "precision_at_50_baseline": 0.26,
  "precision_at_50_model": 0.4,
  "base_rate_bottom_quartile": 0.25992555831265507,
  "n_recommendations": 2041
}


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.